In [ ]:
 #Jupyter Notebook

# Importando as bibliotecas necessárias
import os
import json
import subprocess
import pandas as pd
import seaborn as sns
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import EcoSimpy


# Configurações de exibição
sns.set_theme(style="whitegrid")
pd.options.mode.chained_assignment = None  # Para evitar warnings de cópias de DataFrame


print("Current directory:", os.getcwd())


# Configurando os diretórios
input_dir = Path("runs")
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)


## 2. Leitura e Exploração dos Dados

Carregue os dados usando pandas e explore as primeiras linhas, tipos de dados e estatísticas descritivas.

# Execução da Simulação do El Farol

In [ ]:
from turtle import mode

# Activate virtual environment (if needed, adjust path accordingly)
venv_path = os.path.join(os.getcwd(), "venv")
if os.path.exists(venv_path):
    activate_script = os.path.join(venv_path, "bin", "activate_this.py")
    with open(activate_script) as f:
        exec(f.read(), {'__file__': activate_script})


print("Current directory:", os.getcwd())

config_json_file = "config.json"        

# Get current directory as app_dir
app_dir = os.getcwd()
model_file = "model.json"
scenarios_file = "scenarios.json"


new_sim = EcoSimpy.Simulation(app_dir,
                              clean_run=True)

new_sim.execute_simulation()

print("simulação acabou")
print(os.listdir("runs"))

## Checar depois o que faz
space = list(new_sim.spaces.values())[0]




## 2. Leitura e Exploração dos Dados

Carregue os dados usando pandas e explore as primeiras linhas, tipos de dados e estatísticas descritivas.

In [ ]:
# Encontrar o arquivo de simulação mais recente na pasta 'runs/'
runs_dir = 'runs/'
latest_file = sorted([f for f in os.listdir(runs_dir) if f.endswith('.csv')], reverse=True)[0]
sim_results = pd.read_csv(os.path.join(runs_dir, latest_file))

# Visualizar as primeiras linhas do DataFrame
sim_results.head()

In [ ]:
# Informações gerais sobre os dados
sim_results.info()

# Estatísticas descritivas
sim_results.describe(include='all')

## 3. Pré-processamento dos Dados

Realize limpeza, tratamento de valores ausentes e transformação de variáveis conforme necessário.

In [ ]:
# Verificar valores ausentes
dados_faltantes = sim_results.isnull().sum()
print("Valores ausentes por coluna:\n", dados_faltantes)

# Exemplo de tratamento: remover linhas com valores ausentes (ajuste conforme necessário)
sim_results = sim_results.dropna()


## 4. Análise Exploratória dos Dados

Crie gráficos e tabelas para entender padrões e relações nos dados.

In [ ]:
# Agrupamento semelhante ao R: attendance por run, step, my_play
attendance = sim_results.groupby(['run', 'step', 'my_play']).size().reset_index(name='n')

# Gráfico: Evolução do número de "GOING" ao longo do tempo
plt.figure()
sns.lineplot(data=attendance[attendance['my_play'] == 'GOING'], x='step', y='n')
plt.title('Evolução do número de GOING por step')
plt.xlabel('Step')
plt.ylabel('Número de GOING')
plt.show()

# Gráfico: Relação entre predição e fitness por my_play
plt.figure()
sns.scatterplot(data=sim_results, x='predictor_prediction', y='predictor_fitness', hue='my_play')
plt.title('Predição vs Fitness por my_play')
plt.show()

# Gráfico: Relação entre predição e fitness por selected_predictor
plt.figure()
sns.scatterplot(data=sim_results, x='predictor_prediction', y='predictor_fitness', hue='selected_predictor')
plt.title('Predição vs Fitness por selected_predictor')
plt.show()


In [ ]:
# Densidade das predições por selected_predictor
plt.figure()
sns.kdeplot(data=sim_results, x='predictor_prediction', hue='selected_predictor', common_norm=False)
plt.title('Densidade das predições por selected_predictor')
plt.show()

# Densidade das predições por my_play
plt.figure()
sns.kdeplot(data=sim_results, x='predictor_prediction', hue='my_play', common_norm=False)
plt.title('Densidade das predições por my_play')
plt.show()

# Densidade do log do fitness por selected_predictor
plt.figure()
sns.kdeplot(data=sim_results, x=np.log(sim_results['predictor_fitness']+1e-8), hue='selected_predictor', common_norm=False)
plt.title('Densidade do log do fitness por selected_predictor')
plt.show()

# Densidade do número de GOING/NOT_GOING
plt.figure()
sns.kdeplot(data=attendance, x='n', hue='my_play', common_norm=False)
plt.title('Densidade do número de GOING/NOT_GOING')
plt.show()

# Violino: distribuição de n por my_play
plt.figure()
sns.violinplot(data=attendance, x='my_play', y='n')
plt.title('Distribuição de n por my_play')
plt.show()


## 5. Implementação do Modelo em Python

Implemente o modelo originalmente em R, agora em Python, utilizando as bibliotecas apropriadas.

In [ ]:
# Agrupamento semelhante ao bloco R 'predictors'
predictors = sim_results.groupby(['run', 'step', 'selected_predictor']).agg(
    fitness=('predictor_fitness', 'mean'),
    mean_prediction=('predictor_prediction', 'mean'),
    median_prediction=('predictor_prediction', 'median'),
    n=('predictor_prediction', 'count')
).reset_index()

# Visualização dos resultados do modelo
plt.figure()
sns.lineplot(data=predictors, x='step', y=np.log(predictors['fitness']+1e-8), hue='selected_predictor')
plt.title('Log do fitness médio por step e selected_predictor')
plt.show()

plt.figure()
sns.lineplot(data=predictors, x='step', y='mean_prediction', hue='selected_predictor')
plt.title('Predição média por step e selected_predictor')
plt.show()

plt.figure()
sns.lineplot(data=predictors, x='step', y='median_prediction', hue='selected_predictor')
plt.title('Predição mediana por step e selected_predictor')
plt.show()

plt.figure()
sns.lineplot(data=predictors, x='step', y='n', hue='selected_predictor')
plt.title('Número de predições por step e selected_predictor')
plt.show()


In [ ]:
# Visualização com facet_grid (usando FacetGrid do seaborn)
g = sns.FacetGrid(predictors, row='selected_predictor', hue='selected_predictor', aspect=3, height=2, sharey=False)
g.map(sns.lineplot, 'step', 'mean_prediction')
g.add_legend()
plt.subplots_adjust(top=0.9)
g.fig.suptitle('Predição média por step (facetado por selected_predictor)')
plt.show()

# Ribbon plot (faixa de incerteza)
plt.figure()
for name, group in predictors.groupby('selected_predictor'):
    plt.plot(group['step'], group['mean_prediction'], label=name)
    plt.fill_between(group['step'], group['mean_prediction']-10, group['mean_prediction']+10, alpha=0.2)
plt.title('Predição média com faixa de ±10')
plt.xlabel('Step')
plt.ylabel('Predição média')
plt.legend()
plt.show()


## 6. Avaliação dos Resultados

Avalie o desempenho do modelo utilizando métricas adequadas e compare com os resultados esperados.

In [ ]:
# Exemplo de avaliação: análise descritiva dos resultados
# (Adapte conforme as métricas relevantes para o seu problema)
print('Estatísticas descritivas do fitness por selected_predictor:')
print(predictors.groupby('selected_predictor')['fitness'].describe())


## 7. Visualização dos Resultados

Gere gráficos para ilustrar os resultados do modelo e facilitar a interpretação.

In [ ]:
# Exemplo de visualização: matriz de correlação
plt.figure()
sns.heatmap(sim_results.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Matriz de Correlação das Variáveis Numéricas')
plt.show()

# Visualização de mudança de frequência (semelhante ao bloco R 'changing')
frequencies = attendance.pivot_table(index=['run', 'step'], columns='my_play', values='n', fill_value=0).reset_index()
frequencies.columns.name = None
frequencies['last_going'] = frequencies['GOING'].shift(1)
frequencies['last_not_going'] = frequencies['NOT GOING'].shift(1)

plt.figure()
sns.heatmap(pd.crosstab(frequencies['GOING'], frequencies['last_going']), cmap='Blues')
plt.title('Heatmap: GOING vs last GOING')
plt.show()

plt.figure()
sns.heatmap(pd.crosstab(frequencies['NOT GOING'], frequencies['last_not_going']), cmap='Reds')
plt.title('Heatmap: NOT GOING vs last NOT GOING')
plt.show()
